# SQL LIKE Search

Mục đích: So sánh SQL LIKE và Vector Search (Milvus) trong tìm kiếm ứng viên.

---
## Pipeline
HuggingFace Dataset -> Clean -> SQLite -> SQL LIKE search -> So sánh với Milvus


In [1]:
!pip install datasets tabulate --quiet
print("Libraries installed")

Libraries installed


In [2]:
from datasets import load_dataset

jd_data = load_dataset("lang-uk/recruitment-dataset-job-descriptions-english", split="train[:500]")
cv_data = load_dataset("lang-uk/recruitment-dataset-candidate-profiles-english", split="train[:300]")

print(f"JDs: {len(jd_data)}, CVs: {len(cv_data)}")

JDs: 500, CVs: 300


In [3]:
import re

def clean_text(text):
    if not text or not isinstance(text, str): return ""
    return re.sub(r'\s+', ' ', text).strip()

def truncate_text(text, max_length=500):
    if len(text) <= max_length: return text
    truncated = text[:max_length]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] if last_space > 0 else truncated

cleaned_jd = []
for i, item in enumerate(jd_data):
    description = clean_text(item.get("Long Description"))
    if len(description) >= 20:
        cleaned_jd.append({
            "id": i + 1,
            "position": clean_text(item.get("Position")) or "Unknown",
            "description": truncate_text(description, 2000),
            "company": clean_text(item.get("Company Name")) or "Unknown",
            "keyword": clean_text(item.get("Primary Keyword")) or "",
            "exp_years": clean_text(str(item.get("Exp Years") or "")),
        })

cleaned_cv = []
for i, item in enumerate(cv_data):
    cv_text = clean_text(item.get("CV"))
    if len(cv_text) >= 20:
        cleaned_cv.append({
            "id": i + 1,
            "position": clean_text(item.get("Position")) or "Unknown",
            "cv_text": truncate_text(cv_text, 2000),
            "highlights": clean_text(item.get("Highlights")) or "",
            "keyword": clean_text(item.get("Primary Keyword")) or "",
            "exp_years": clean_text(str(item.get("Experience Years") or "")),
            "looking_for": clean_text(item.get("Looking For")) or "",
        })

print(f"Cleaned JD: {len(cleaned_jd)}")
print(f"Cleaned CV: {len(cleaned_cv)}")
print("Note: No vector embedding created. SQL uses raw text.")

Cleaned JD: 500
Cleaned CV: 300
Note: No vector embedding created. SQL uses raw text.


In [4]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("""
    CREATE TABLE job_descriptions (
        id INTEGER PRIMARY KEY, position TEXT, description TEXT,
        company TEXT, keyword TEXT, exp_years TEXT
    )
""")
cur.execute("""
    CREATE TABLE cvs (
        id INTEGER PRIMARY KEY, position TEXT, cv_text TEXT,
        highlights TEXT, keyword TEXT, exp_years TEXT, looking_for TEXT
    )
""")

cur.executemany("INSERT INTO job_descriptions VALUES (:id, :position, :description, :company, :keyword, :exp_years)", cleaned_jd)
cur.executemany("INSERT INTO cvs VALUES (:id, :position, :cv_text, :highlights, :keyword, :exp_years, :looking_for)", cleaned_cv)
conn.commit()

cur.execute("CREATE INDEX idx_cv_position ON cvs(position)")
cur.execute("CREATE INDEX idx_cv_keyword ON cvs(keyword)")
cur.execute("CREATE INDEX idx_jd_position ON job_descriptions(position)")

print("SQLite DB in-memory created.")
print("Note: Index on TEXT doesn't help with LIKE '%keyword%'. Full Table Scan is used.")

SQLite DB in-memory created.
Note: Index on TEXT doesn't help with LIKE '%keyword%'. Full Table Scan is used.


In [5]:
import time
from tabulate import tabulate

def extract_keywords(query: str) -> list[str]:
    STOP_WORDS = {"with", "and", "the", "for", "in", "of", "to", "a", "an", "is", "are", "that", "this", "have", "has", "been", "will", "can", "from", "or", "on", "at", "by", "who", "our"}
    words = re.findall(r'[a-zA-Z]+', query.lower())
    return [w for w in words if len(w) >= 3 and w not in STOP_WORDS]

def sql_like_search_cv(query: str, limit: int = 5) -> dict:
    keywords = extract_keywords(query)
    start_time = time.perf_counter()
    if not keywords:
        return {"results": [], "keywords": [], "strategy": "none", "elapsed_ms": 0, "query": query}
    
    and_conditions = " AND ".join(f"(cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' OR highlights LIKE '%{kw}%' OR keyword LIKE '%{kw}%')" for kw in keywords)
    match_score_expr = " + ".join(f"(CASE WHEN cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' OR highlights LIKE '%{kw}%' THEN 1 ELSE 0 END)" for kw in keywords)
    
    and_sql = f"SELECT id, position, keyword, cv_text, highlights, ({match_score_expr}) AS match_score FROM cvs WHERE {and_conditions} ORDER BY match_score DESC LIMIT {limit}"
    cur.execute(and_sql)
    results = [dict(row) for row in cur.fetchall()]
    strategy = "AND"
    
    if not results:
        strategy = "OR"
        or_conditions = " OR ".join(f"(cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' OR highlights LIKE '%{kw}%' OR keyword LIKE '%{kw}%')" for kw in keywords)
        or_sql = f"SELECT id, position, keyword, cv_text, highlights, ({match_score_expr}) AS match_score FROM cvs WHERE {or_conditions} ORDER BY match_score DESC LIMIT {limit}"
        cur.execute(or_sql)
        results = [dict(row) for row in cur.fetchall()]
    
    return {
        "query": query, "keywords": keywords, "strategy": strategy,
        "results": results, "elapsed_ms": (time.perf_counter() - start_time) * 1000,
    }

def print_search_results(search_result: dict):
    q, kws, stg, ms, res = search_result["query"], search_result["keywords"], search_result["strategy"], search_result["elapsed_ms"], search_result["results"]
    print(f"\nQuery: {q}\nKeywords: {kws}\nStrategy: SQL LIKE [{stg}]\nTime: {ms:.2f} ms\nResults: {len(res)}")
    
    if not res:
        print("No candidates found.")
        return
    
    table_data = [[f"#{i}", row["position"][:30], f"{row['match_score']}/{len(kws)}", row.get("keyword", "")[:40], row.get("cv_text", "")[:80] + "..."] for i, row in enumerate(res, 1)]
    print(tabulate(table_data, headers=["#", "Position", "Match Score", "Keyword", "Snippet"], tablefmt="simple"))

print("Search function ready.")

Search function ready.


In [6]:
queries = [
    "Python backend developer with Django and REST API experience",
    "server-side engineer who builds web APIs and database integrations",
    "someone who thrives in fast-paced small teams with rapid iteration",
    "strong communicator who can lead cross-functional projects",
    "I need a data person who understands both the business side and technical implementation",
]

all_results = []
for q in queries:
    result = sql_like_search_cv(q, limit=5)
    all_results.append(result)
    print_search_results(result)


Query: Python backend developer with Django and REST API experience
Keywords: ['python', 'backend', 'developer', 'django', 'rest', 'api', 'experience']
Strategy: SQL LIKE [OR]
Time: 6.41 ms
Results: 5
#    Position      Match Score    Keyword    Snippet
---  ------------  -------------  ---------  -----------------------------------------------------------------------------------
#1   1C developer  3/7            Flutter    1 am an 1C developer. I deployed an 1C to typographical factory in Ukraine. Also...
#2   1C Developer  3/7            Other      Perfect knowledge of 1C:Enterprise Platform. Good understanding of OOP, Java Cor...
#3   1С Developer  3/7            Other      1C 8.2, 8.3 programming, reports, processing, SKD. UPP, custom configurations, m...
#4   2D animator   3/7            Unity      As an animator, as part of the art team & developers, I took part in the creatio...
#5   2d artist     3/7            Artist     I finished 8 years of study with excellent grades. Duri

In [7]:
print("\n--- SQL LIKE vs MILVUS ---")

summary_data = []
empty_count = 0
fallback_count = 0

for r in all_results:
    n = len(r["results"])
    if n == 0: empty_count += 1
    if r["strategy"] == "OR": fallback_count += 1
    avg_score = sum(x["match_score"] for x in r["results"]) / n if n > 0 else 0
    status = "EMPTY" if n == 0 else ("Fallback OR" if r["strategy"] == "OR" else "AND")
    summary_data.append([r["query"][:30] + "...", status, n, f"{avg_score:.1f}/{len(r['keywords'])}", f"{r['elapsed_ms']:.1f}ms"])

print(tabulate(summary_data, headers=["Query", "Strategy", "Results", "Avg Score", "Time"], tablefmt="simple"))

print("\n--- CONCLUSION ---")
print(f"SQL LIKE Empty Results: {empty_count}/5")
print(f"SQL LIKE Fallback OR: {fallback_count}/5")
print("SQL LIKE fails on semantic meaning and synonyms. Vector Search solves this.")


--- SQL LIKE vs MILVUS ---
Query                              Strategy       Results  Avg Score    Time
---------------------------------  -----------  ---------  -----------  ------
Python backend developer with ...  Fallback OR          5  3.0/7        6.4ms
server-side engineer who build...  Fallback OR          5  2.2/8        4.8ms
someone who thrives in fast-pa...  Fallback OR          5  2.4/8        5.5ms
strong communicator who can le...  Fallback OR          5  2.4/6        3.3ms
I need a data person who under...  Fallback OR          5  3.2/9        3.9ms

--- CONCLUSION ---
SQL LIKE Empty Results: 0/5
SQL LIKE Fallback OR: 5/5
SQL LIKE fails on semantic meaning and synonyms. Vector Search solves this.
